# CODLING Model Architecture
============================

This notebook explores the CODLING SSM architecture, showing model configuration, structure, and key differences from attention-based models.

**Contents:**
- Import and display model configuration
- Build a small model (130M params)
- Show model architecture summary
- Test forward pass with random data
- Demonstrate SSM vs Attention differences
- Visualize model components

---

## 1. Setup and Imports
Import required packages and configure the environment.

In [ ]:
import sys
sys.path.insert(0, '/content')

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from typing import Dict, Any

print("=" * 50)
print("IMPORTS COMPLETE")
print("=" * 50)
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

## 2. Model Configuration
Define and explore the CODLING model configuration.

In [ ]:
# Define configuration directly since we may not have the full repo
from dataclasses import dataclass, field
from typing import Optional

@dataclass
class CodlingConfig:
    """Configuration for CODLING SSM Language Model."""
    
    # Model architecture
    vocab_size: int = 50257
    hidden_size: int = 768
    num_hidden_layers: int = 24
    num_attention_heads: int = 12
    intermediate_size: int = 3072
    
    # SSM configuration
    ssm_type: str = "mamba"  # "mamba", "s4", "mamba_v2"
    use_hyena: bool = False
    use_linear_attn: bool = False
    d_state: int = 128
    d_conv: int = 4
    expand_factor: int = 2
    
    # Position embeddings
    max_position_embeddings: int = 2048
    use_rope: bool = True
    rope_theta: float = 10000.0
    
    # Normalization
    rms_norm_eps: float = 1e-6
    
    # Initialization
    initializer_range: float = 0.02
    
    # Training
    use_cache: bool = True
    pad_token_id: int = 0
    bos_token_id: int = 1
    eos_token_id: int = 2

# Model size presets
MODEL_SIZES = {
    "130M": {
        "hidden_size": 768,
        "num_hidden_layers": 24,
        "intermediate_size": 3072,
        "d_state": 128,
    },
    "350M": {
        "hidden_size": 1024,
        "num_hidden_layers": 24,
        "intermediate_size": 4096,
        "d_state": 128,
    },
    "760M": {
        "hidden_size": 1280,
        "num_hidden_layers": 32,
        "intermediate_size": 5120,
        "d_state": 256,
    },
}

# Create small model config (130M)
config = CodlingConfig(
    hidden_size=768,
    num_hidden_layers=24,
    intermediate_size=3072,
    d_state=128,
    ssm_type="mamba",
)

print("=" * 50)
print("CODLING CONFIGURATION (130M)")
print("=" * 50)
for key, value in config.__dict__.items():
    print(f"  {key}: {value}")

## 3. Estimate Model Parameters
Calculate the total number of parameters in the model.

In [ ]:
def estimate_params(config: CodlingConfig) -> Dict[str, int]:
    """Estimate model parameters by component."""
    
    params = {}
    
    # Embeddings
    params['token_embedding'] = config.vocab_size * config.hidden_size
    params['position_embedding'] = config.max_position_embeddings * config.hidden_size
    
    # Per-layer params
    layer_params = {}
    
    # Layer Norm
    layer_params['input_norm'] = config.hidden_size
    layer_params['post_attn_norm'] = config.hidden_size
    
    # SSM (Mamba-style)
    ssm_expanded = config.hidden_size * config.expand_factor * 2
    ssm_state = config.d_state
    layer_params['ssm_in_proj'] = ssm_expanded
    layer_params['ssm_x_proj'] = ssm_expanded // 2 * config.d_state
    layer_params['ssm_dt_proj'] = config.expand_factor
    layer_params['ssm_A'] = config.d_state * config.d_state
    layer_params['ssm_out_proj'] = config.hidden_size * config.expand_factor * 2
    
    # MLP (SwiGLU)
    layer_params['mlp_gate_proj'] = config.hidden_size * config.intermediate_size
    layer_params['mlp_up_proj'] = config.hidden_size * config.intermediate_size
    layer_params['mlp_down_proj'] = config.intermediate_size * config.hidden_size
    
    total_layer = sum(layer_params.values())
    params['per_layer'] = total_layer
    params['all_layers'] = total_layer * config.num_hidden_layers
    
    # Final layer norm
params['final_norm'] = config.hidden_size
    
    # LM head
params['lm_head'] = config.hidden_size * config.vocab_size
    
    return params

params = estimate_params(config)

print("=" * 50)
print("PARAMETER ESTIMATION")
print("=" * 50)

total = sum(params.values())
for name, count in params.items():
    print(f"  {name:25s}: {count:>12,} ({count/total*100:>5.1f}%)")

print("-" * 50)
print(f"  {'TOTAL':25s}: {total:>12,} ({total/1e6:.1f}M)")

## 4. Build Simplified Model
Create a minimal CODLING-like model for demonstration.

In [ ]:
class SimplifiedSSMBlock(nn.Module):
    """Simplified SSM block (Mamba-style)."""
    
    def __init__(self, hidden_size: int, d_state: int = 128, d_conv: int = 4):
        super().__init__()
        self.hidden_size = hidden_size
        self.d_state = d_state
        self.d_conv = d_conv
        self.expand = 2
        
        # Input projection
        self.in_proj = nn.Linear(hidden_size, hidden_size * self.expand * 2, bias=False)
        
        # Convolutional layer
        self.conv1d = nn.Conv1d(
            in_channels=hidden_size * self.expand,
            out_channels=hidden_size * self.expand,
            kernel_size=d_conv,
            padding=d_conv - 1,
            groups=hidden_size * self.expand,
        )
        
        # SSM parameters
        self.x_proj = nn.Linear(hidden_size * self.expand, d_state, bias=False)
        self.dt_proj = nn.Linear(d_state, hidden_size * self.expand, bias=True)
        
        # State space matrix A (learned)
        self.A_log = nn.Parameter(torch.randn(d_state, hidden_size * self.expand))
        self.D = nn.Parameter(torch.ones(hidden_size * self.expand))
        
        # Output projection
        self.out_proj = nn.Linear(hidden_size * self.expand, hidden_size, bias=False)
        
        # Layer norms
        self.norm = nn.RMSNorm(hidden_size)
        
    def forward(self, x):
        """Forward pass with selective scan."""
        # x: (batch, seq_len, hidden)
        residual = x
        x = self.norm(x)
        
        # Input projection
        xz = self.in_proj(x)  # (batch, seq, 2*expand*hidden)
        x_inner, z = xz.chunk(2, dim=-1)
        
        # Convolution (simplified - just use linear for demo)
        x_conv = self.conv1d(x_inner.transpose(1, 2))[:, :, :x_inner.size(1)]
        x_conv = x_conv.transpose(1, 2)
        x_conv = F.silu(x_conv)
        
        # SSM (simplified - use dense for demo)
        ssm_input = x_conv
        ssm_params = self.x_proj(ssm_input)
        dt = self.dt_proj(ssm_params)
        dt = F.softplus(dt)
        
        # Simplified SSM computation
        A = -torch.exp(self.A_log.float())
        y = ssm_input * torch.sigmoid(dt) + self.D * ssm_input
        
        # Gating
        y = y * F.silu(z)
        
        # Output projection
        out = self.out_proj(y)
        
        return out + residual


class SimplifiedCodling(nn.Module):
    """Simplified CODLING model for demonstration."""
    
    def __init__(self, config: CodlingConfig):
        super().__init__()
        self.config = config
        
        # Token embeddings
        self.token_embeddings = nn.Embedding(config.vocab_size, config.hidden_size)
        
        # Position embeddings
        self.position_embeddings = nn.Embedding(config.max_position_embeddings, config.hidden_size)
        
        # SSM blocks
        self.layers = nn.ModuleList([
            SimplifiedSSMBlock(
                config.hidden_size,
                config.d_state,
                config.d_conv,
            )
            for _ in range(config.num_hidden_layers)
        ])
        
        # Final norm
        self.final_norm = nn.RMSNorm(config.hidden_size)
        
        # LM head
        self.lm_head = nn.Linear(config.hidden_size, config.vocab_size, bias=False)
        
        # Tie weights
        self.lm_head.weight = self.token_embeddings.weight
        
    def forward(self, input_ids, attention_mask=None):
        """Forward pass."""
        batch_size, seq_len = input_ids.shape
        
        # Embeddings
        hidden_states = self.token_embeddings(input_ids)
        
        # Position IDs
        position_ids = torch.arange(seq_len, device=input_ids.device)
        position_ids = position_ids.unsqueeze(0).expand(batch_size, -1)
        hidden_states = hidden_states + self.position_embeddings(position_ids)
        
        # Apply SSM layers
        for layer in self.layers:
            hidden_states = layer(hidden_states)
        
        # Final norm
        hidden_states = self.final_norm(hidden_states)
        
        # LM head
        logits = self.lm_head(hidden_states)
        
        return logits

print("Defining model classes... (importing F)")
import torch.nn.functional as F
print("✅ Model classes defined!")

## 5. Create and Test Model
Instantiate the model and run a forward pass.

In [ ]:
# Create a smaller model for testing (reduce layers for speed)
test_config = CodlingConfig(
    vocab_size=50257,
    hidden_size=256,
    num_hidden_layers=4,  # Reduced for quick testing
    intermediate_size=1024,
    d_state=64,
    max_position_embeddings=512,
)

print("=" * 50)
print("CREATING TEST MODEL")
print("=" * 50)
print(f"Config: hidden_size={test_config.hidden_size}, layers={test_config.num_hidden_layers}")

# Build model
model = SimplifiedCodling(test_config)
model = model.to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

## 6. Forward Pass Test
Test the model with random input data.

In [ ]:
# Generate random input
batch_size = 2
seq_len = 32

input_ids = torch.randint(0, test_config.vocab_size, (batch_size, seq_len), device=device)

print("=" * 50)
print("FORWARD PASS TEST")
print("=" * 50)
print(f"Input shape: {input_ids.shape}")

# Forward pass
model.eval()
with torch.no_grad():
    torch.cuda.reset_peak_memory_stats() if torch.cuda.is_available() else None
    
    logits = model(input_ids)
    
    if torch.cuda.is_available():
        mem_allocated = torch.cuda.memory_allocated() / 1024**2
        mem_reserved = torch.cuda.memory_reserved() / 1024**2
        print(f"GPU memory allocated: {mem_allocated:.2f} MB")
        print(f"GPU memory reserved: {mem_reserved:.2f} MB")

print(f"Output logits shape: {logits.shape}")
print(f"Output shape expected: ({batch_size}, {seq_len}, {test_config.vocab_size})")

# Test loss computation
labels = input_ids[:, 1:].contiguous()
logits_for_loss = logits[:, :-1, :].contiguous()
loss_fn = nn.CrossEntropyLoss()
loss = loss_fn(
    logits_for_loss.view(-1, test_config.vocab_size),
    labels.view(-1)
)
print(f"\nLoss (random): {loss.item():.4f}")

## 7. SSM vs Attention: Conceptual Comparison
Visualize the differences between SSM and attention mechanisms.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# 1. Attention Pattern (Full quadratic)
seq_len = 32
attn_pattern = np.ones((seq_len, seq_len))
attn_pattern = np.triu(attn_pattern, k=0)  # Causal mask

axes[0].imshow(attn_pattern, cmap='Blues', aspect='auto')
axes[0].set_title('Standard Attention\n(O(n²) complexity)', fontsize=12)
axes[0].set_xlabel('Position j')
axes[0].set_ylabel('Position i')
axes[0].set_xticks([0, 16, 31])
axes[0].set_yticks([0, 16, 31])

# 2. SSM Pattern (Linear)
ssm_pattern = np.zeros((seq_len, seq_len))
for i in range(seq_len):
    ssm_pattern[i, :i+1] = np.exp(-0.1 * (i - np.arange(i+1)))  # Decay

axes[1].imshow(ssm_pattern, cmap='Greens', aspect='auto')
axes[1].set_title('State Space Model (SSM)\n(O(n) complexity)', fontsize=12)
axes[1].set_xlabel('State dimension')
axes[1].set_ylabel('Position i')
axes[1].set_xticks([0, 16, 31])
axes[1].set_yticks([0, 16, 31])

# 3. Complexity comparison
seq_lengths = [128, 512, 2048, 8192, 32768]
attn_complexity = [n**2 for n in seq_lengths]
ssm_complexity = [n * 128 for n in seq_lengths]  # d_state = 128

axes[2].plot(seq_lengths, attn_complexity, 'b-o', label='Attention O(n²)')
axes[2].plot(seq_lengths, ssm_complexity, 'g-s', label='SSM O(n·d)')
axes[2].set_xscale('log')
axes[2].set_yscale('log')
axes[2].set_xlabel('Sequence Length')
axes[2].set_ylabel('Compute Operations')
axes[2].set_title('Complexity Comparison', fontsize=12)
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/tmp/ssm_vs_attention.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ Visualization saved to /tmp/ssm_vs_attention.png")

## 8. Memory Usage Analysis
Analyze GPU memory usage for different sequence lengths.

In [ ]:
if torch.cuda.is_available():
    seq_lengths = [32, 64, 128, 256, 512]
    memory_usage = []
    
    print("=" * 50)
print("MEMORY USAGE BY SEQUENCE LENGTH")
print("=" * 50)

    model.eval()
    with torch.no_grad():
        for seq_len in seq_lengths:
            torch.cuda.reset_peak_memory_stats()
            
            input_ids = torch.randint(0, test_config.vocab_size, (1, seq_len), device=device)
            logits = model(input_ids)
            
            mem_mb = torch.cuda.max_memory_allocated() / 1024**2
            memory_usage.append(mem_mb)
            
            print(f"Seq len {seq_len:4d}: {mem_mb:>8.2f} MB")
    
    # Plot
    plt.figure(figsize=(8, 5))
    plt.plot(seq_lengths, memory_usage, 'bo-')
    plt.xlabel('Sequence Length')
    plt.ylabel('GPU Memory (MB)')
plt.title('Memory Usage vs Sequence Length')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('/tmp/memory_usage.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("CUDA not available - skipping GPU memory test")

## 9. Model Architecture Summary
Display model structure and summary.

In [ ]:
print("=" * 60)
print("CODLING MODEL ARCHITECTURE SUMMARY")
print("=" * 60)

print("""
CODLING is a State Space Model (SSM)-based language model.

Key Components:
----------------
1. Token Embeddings - Maps vocabulary to hidden dimension
2. Position Embeddings - Adds positional information (RoPE)
3. SSM Blocks (xN) - Core state space computation
   - Input projection with gating
   - Depthwise convolution
   - Selective state space (Mamba-style)
   - Output projection
4. RMSNorm - Stable normalization (LLaMA-style)
5. SwiGLU MLP - Feed-forward network
6. LM Head - Output projection to vocabulary

SSM Mathematics:
----------------
  h_t = A @ h_{t-1} + B @ x_t    (state update)
  y_t = C @ h_t + D @ x_t        (output)

Where:
  - A: state-to-state transition matrix
  - B: input-to-state projection
  - C: state-to-output projection
  - D: skip connection (discrete convolution)

Advantages of SSM:
-------------------
✓ Linear complexity O(n) vs quadratic O(n²) for attention
✓ Efficient long-range modeling without attention
✓ Parallelizable computation during training
✓ Constant KV state size (no KV cache growth)
✓ Strong performance on code generation tasks
""")

print("=" * 60)
print("Next: Open 03_training.ipynb to start training!")
print("=" * 60)